# PHYS 481 — Unit 1, Day 2
# Building the derivative operator

**Names:** _______________________________________


| | |
|---|---|
| **Computers** | **Yes.** This is a hands-on class. |
| **AI / LLM assistants** | **No.** Turn off inline autocomplete (Copilot, Cursor tab-completion) before you start. |
| **Partners** | Strongly encouraged. Talk out loud. |
| **Hand in** | Save and submit this notebook at the end of class. Graded attempted / not attempted. |

### What this worksheet is for

This worksheet steps you through building and testing the core functions required for the assignment.
By the end of this class you will have written and tested `central_diff` — the central function
Assignment 1 is built around — and you will have seen what truncation error and quantization
do to a numerical derivative.

Most cells below already run. Two kinds of gap are marked for you:

* **⭐ YOUR TURN** — write code.
* **✍** — write an answer, in words, in the markdown cell.

Work through the sections in order — each one uses what the ones before it defined.

---
## 0 · ⭐ YOUR TURN — set up the test grid

Everything today is tested against $f(t)=\sin(\omega t)$, because we know that
$f'(t) = \omega\cos(\omega t)$.
That allows us to test our methods against a known-right answer.

In the code block below, functions `f` and `f_prime_exact` are given. You need to write `make_grid`.

**What it does:** `make_grid(n_points, t_max)` returns `n_points` evenly spaced sample times
covering $[0, t_{\max}]$ — both ends included — and the spacing $h$ between them.

Watch out for an off-by-one error: $n$ points
laid end to end span $n-1$ gaps, not $n$. The tests check the value of $h$, so if you divide by the
wrong thing you will hear about it straight away.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

OMEGA = 2.0            # rad/s

def f(t):
    '''Test signal.'''
    return np.sin(OMEGA * t)

def f_prime_exact(t):
    '''Analytic derivative of f -- our source of truth.'''
    return OMEGA * np.cos(OMEGA * t)

def make_grid(n_points, t_max=10.0):
    '''
    n_points uniformly spaced samples covering [0, t_max], both ends included.

    Returns t : the sample times
            h : the spacing between them
    '''
    # ⭐ YOUR CODE HERE -- two lines. Delete the raise below.
    raise NotImplementedError('build the grid, and return the times and the spacing')


# ---- tests: re-run this cell until they all pass ---------------------------
t, h = make_grid(201)
assert len(t) == 201,              f'expected 201 samples, got {len(t)}'
assert np.isclose(t[0], 0.0),      f'the grid should start at 0, starts at {t[0]}'
assert np.isclose(t[-1], 10.0),    f'the grid should end at t_max = 10, ends at {t[-1]}'
assert np.allclose(np.diff(t), h), 'the spacing is not uniform, or h is not that spacing'
assert np.isclose(h, 0.05),        f'201 points covering [0, 10] means h = 0.05, not {h}'
print(f'{len(t)} samples, h = {h:.4f} s   -- grid OK')

---
## 1 · ⭐ YOUR TURN — write `central_diff`

This is the function Assignment 1 is built around. Everything else today uses it.

**What it does.** `central_diff(y, h, stencil=3)` takes an array `y` of samples spaced `h` apart and
returns the numerical derivative at each interior sample, using the three-point central difference
you derived on Tuesday:

$$f'(x_n) \approx \frac{f(x_{n+1}) - f(x_{n-1})}{2h}$$

Sample $n$ needs its two neighbours, so the first and last samples get no value. The array that
comes back is therefore **shorter than `y` by two**: `len(y) - 2` for the three-point formula, and
`len(y) - (stencil - 1)` in general.

**What to write.** One line, inside the `if stencil == 3:` branch. Replace the comment *and* the
`raise NotImplementedError` under it — that `raise` is a placeholder that makes the function fail
loudly while it is still empty, so delete it once your line is in.

Two rules:

* **Slicing, not a Python loop.** Operate on the whole array at once.
* **Not `np.gradient` or `np.diff`.** Writing it is the exercise.

The two tests at the bottom of the cell run every time you run the cell. Edit, run, read the
message, edit again — do not move on until both print `OK`. The second test is the quadratic you
worked through by hand on Tuesday: for $a_i = i^2+1$ with $h=1$ the exact derivative is $2i$, and
the three-point formula gets it exactly right.

In [ ]:
def central_diff(y, h, stencil=3):
    '''
    Numerical derivative of evenly spaced samples, by a central difference.

    y       : 1-D array of samples
    h       : spacing between samples
    stencil : 3, 5 or 7  (only 3 needs to work for now)

    Returns an array of length len(y) - (stencil - 1).
    '''
    if stencil == 3:
        # ⭐ YOUR CODE HERE -- one line, using slices. Delete the raise below.
        raise NotImplementedError('write the 3-point central difference')
    raise ValueError(f'stencil {stencil} not implemented yet')


# ---- tests: re-run this cell until both print OK ---------------------------
y_test = f(t)
d = central_diff(y_test, h, 3)
assert len(d) == len(y_test) - 2, f'wrong length: expected {len(y_test)-2}, got {len(d)}'
print('length OK   :', len(d))

a   = np.array([1.0, 2.0, 5.0, 10.0, 17.0])      # a[i] = i**2 + 1, spacing 1, derivative 2i
got = central_diff(a, 1.0, 3)
assert np.allclose(got, [2.0, 4.0, 6.0]), f'expected [2. 4. 6.], got {got}'
print('quadratic OK:', got)

---
## 2 · Compare against the exact derivative

The cell below is **given** — it plots your numerical derivative against the analytic one, and
plots the difference.

Notice `t[1:-1]`: the derivative array is shorter than the data, so the time axis has to be trimmed
to match. Getting this wrong is the single most common bug in Assignment 1.

In [ ]:
d_num = central_diff(f(t), h, 3)

fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
ax[0].plot(t, f_prime_exact(t), 'k-',  lw=2, label='exact')
ax[0].plot(t[1:-1], d_num, 'r--', lw=1.5, label='3-point')
ax[0].set_xlabel('t [s]'); ax[0].set_ylabel("f '(t)"); ax[0].legend(); ax[0].set_title('derivative')

ax[1].plot(t[1:-1], d_num - f_prime_exact(t[1:-1]), 'b-')
ax[1].set_xlabel('t [s]'); ax[1].set_ylabel('numerical - exact'); ax[1].set_title('error')
plt.tight_layout(); plt.show()

print(f'max |error| = {np.max(np.abs(d_num - f_prime_exact(t[1:-1]))):.3e}')

**✍ Q2.1** The left plot looks like a perfect match. The right plot shows it is not. Roughly how big
is the error compared with the signal itself (which has amplitude $\omega = 2$)? *(one line)*

*Your answer:*

**✍ Q2.2** Why does the error curve have the shape it does — why is it largest in some places and
near zero in others? *(Two or three sentences. Hint: the leading error term is proportional
to $f'''$.)*

*Your answer:*

---
## 3 · ⭐ YOUR TURN — measure the rate of convergence

Section 2 was already a real test, and a strong one: you compared your derivative against an
analytic answer known independently, and the two agreed to about one part in $10^3$. That is much
better evidence than a curve that merely looks smooth.

What it does not tell you is *which* scheme you implemented. A first-order forward difference would
also have produced a line sitting on top of the exact one at this $h$. Second-order accuracy is a
statement about how the error behaves **as $h$ shrinks**, so checking it means varying $h$ and
watching the error respond. That experiment is the **convergence test**, and it is the standard way
to establish that a numerical scheme is doing what you think it is.

From Worksheet 1: the three-point central difference is second-order accurate, so halving $h$
should cut the error by a factor of ______.

You write `max_error(n)`: it builds a grid of `n` points, differentiates $f$ on it, and returns `h`
together with the largest error anywhere on the grid. The subtlety is the one from Section 2 —
`central_diff` hands back a shorter array than it was given, so the exact derivative has to be
trimmed to match before you subtract. `k`, the number of points lost at each end, is given.

The grid always spans the same 10 seconds, so **doubling $n-1$ halves $h$ exactly**. Once the test
passes, run it for $n = 101$, then 201, 401 and 801, and record what you get.

In [ ]:
def max_error(n_points, stencil=3):
    '''Largest absolute error of central_diff on an n_points grid. Returns (h, max error).'''
    k = (stencil - 1) // 2                      # points lost at each end
    # ⭐ YOUR CODE HERE -- three lines: build the grid, differentiate f on it, and
    #    return (h, the largest absolute error against the exact derivative).
    #    Trim the exact derivative by k at each end so the two arrays line up.
    raise NotImplementedError


# ---- test: Section 2 already printed this number ---------------------------
h_chk, e_chk = max_error(201)
assert np.isclose(h_chk, 0.05), f'h should be 0.05 on a 201-point grid, got {h_chk}'
assert np.isclose(e_chk, 3.33e-3, rtol=2e-2), f'expected about 3.33e-03, got {e_chk:.3e}'
print('matches the number Section 2 printed  OK')

for n in [101]:                # ⭐ YOUR TURN -- add 201 and 401 to this list
    hn, err = max_error(n)
    print(f'n = {n:5d}    h = {hn:.5f}    max error = {err:.3e}')

Fill this in from the printout. The last column is the previous error divided by this one.

| $n$ | $h$ | max error | error ratio |
|---|---|---|---|
| 101 | | | — |
| 201 | | | |
| 401 | | | |

**✍ Q3.1** Do the ratios match your prediction? *(one line)*

*Your answer:*

---
## 4 · ⭐ YOUR TURN — the 7-point stencil

Cancelling more terms of the Taylor expansion buys a higher order of accuracy, at the cost of a
wider stencil:

$$f'(x)\approx\frac{f_{n-2}-8f_{n-1}+8f_{n+1}-f_{n+2}}{12h}\qquad\text{(5-point, 4th order)}$$

$$f'(x)\approx\frac{-f_{n-3}+9f_{n-2}-45f_{n-1}+45f_{n+1}-9f_{n+2}+f_{n+3}}{60h}\qquad\text{(7-point, 6th order)}$$

The cell below redefines `central_diff` with both. **The 5-point branch is written for you** — read
it against the formula above and satisfy yourself that each slice picks out the right neighbour.
Then write the 7-point branch the same way.

**Two checks you can do in your head before running:**

1. Every slice on the line must have the **same length**, `len(y) - (stencil-1)`. For the 7-point
   formula each slice drops six elements: `y[:-6]`, `y[1:-5]`, and so on.
2. The weights must **sum to zero** — $-1+9-45+45-9+1=0$ — because the derivative of a constant is
   zero. That is what the test below checks.

In [ ]:
def central_diff(y, h, stencil=3):
    '''Numerical derivative by a central finite difference. stencil = 3, 5 or 7.'''
    if stencil == 3:
        return (y[2:] - y[:-2]) / (2 * h)
    if stencil == 5:
        return (y[:-4] - 8*y[1:-3] + 8*y[3:-1] - y[4:]) / (12 * h)
    if stencil == 7:
        # ⭐ YOUR CODE HERE -- one line, six slices. Delete the raise below.
        raise NotImplementedError('write the 7-point central difference')
    raise ValueError('stencil must be 3, 5 or 7')

# sanity: a constant must differentiate to zero for every stencil
const = np.ones(20)
for s in (3, 5, 7):
    assert np.allclose(central_diff(const, 0.1, s), 0.0), f'stencil {s} fails the constant test'
    print(f'stencil {s}: constant test OK, output length {len(central_diff(const,0.1,s))} (expect {20-(s-1)})')

The next cell repeats your Section 3 experiment for all three stencils and plots the result — it is
given, so just run it and read the output.

**Predict the ratios first:** 3-point ÷ ____, 5-point ÷ ____, 7-point ÷ ____.

In [ ]:
ORDER = {3: 2, 5: 4, 7: 6}

def convergence(stencil, n_list=(51, 101, 201, 401, 801, 1601, 3201, 6401)):
    hs, errs = [], []
    for n in n_list:
        hn, e = max_error(n, stencil)
        hs.append(hn); errs.append(e)
    return np.array(hs), np.array(errs)

plt.figure(figsize=(5.5, 4))
for s, style in [(3, 'o-'), (5, 's-'), (7, '^-')]:
    hs, errs = convergence(s)
    ratios   = errs[:-1] / errs[1:]
    print(f'{s}-point: ratios per halving = ' + ' '.join(f'{r:6.1f}' for r in ratios)
          + f'   (predicted {2**ORDER[s]})')
    plt.loglog(hs, errs, style, label=f'{s}-point')
plt.xlabel('h [s]'); plt.ylabel('max error'); plt.legend(); plt.grid(True, which='both', alpha=.3)
plt.title('convergence'); plt.tight_layout(); plt.show()

**✍ Q4.1** Do the measured ratios match 4, 16 and 64? *(one line)*

*Your answer:*

**✍ Q4.2** Look at the 7-point line at the smallest $h$. Does it keep improving? If not, what has it
run into? *(two or three sentences)*

*Your answer:*

---
## 5 · Quantization

Stored numbers have finite precision. The satellite positions in Assignment 1 are recorded to a
fixed number of decimals, which means the data arrive as a **staircase**.

This cell is given. Run it, look carefully at the right-hand panel, and then read the three
numbers printed underneath.

In [ ]:
STENCIL = 3                             # the figure below is drawn with the 3-point stencil

t_q, h_q = make_grid(401)
y_true   = f(t_q)
y_quant  = np.round(y_true, 1)          # keep 1 decimal -- a coarse staircase

k       = (STENCIL - 1) // 2
t_c     = t_q[k:-k]                     # time axis trimmed to match the derivative
d_true  = central_diff(y_true,  h_q, STENCIL)
d_quant = central_diff(y_quant, h_q, STENCIL)

fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
ax[0].plot(t_q[:60], y_true[:60], 'k-', label='true')
ax[0].plot(t_q[:60], y_quant[:60], 'r.-', ms=4, lw=.8, label='quantized')
ax[0].set_title('the signal (zoomed)'); ax[0].set_xlabel('t [s]'); ax[0].legend()

ax[1].plot(t_c, d_true, 'k-', label='from true data')
ax[1].plot(t_c, d_quant, 'r-', lw=.8, label='from quantized data')
ax[1].set_title(f'the derivative ({STENCIL}-point)'); ax[1].set_xlabel('t [s]'); ax[1].legend()
plt.tight_layout(); plt.show()

print(f'quantization step         : {0.1:.3f}   ({int(2/0.1)} levels across the signal)')
print(f'3-point spike ~ step / 2h : {0.1/(2*h_q):.2f}')
print()
print('extra scatter in the derivative, by stencil:')
for s in (3, 5, 7):
    ks = (s - 1) // 2
    sc = np.std(central_diff(y_quant, h_q, s) - central_diff(y_true, h_q, s))
    print(f'   {s}-point: {sc:.3f}   =  {100*sc/OMEGA:3.0f}% of the derivative amplitude')

**✍ Q5.1** Zoom in on the left panel: the signal is now a visible staircase, but the steps are still
small compared with its amplitude — it is recognisably the same sine wave. The derivative is not
recognisable at all. Explain in one sentence why differencing turns a small defect into a large one.

*Your answer:*

**✍ Q5.2** The last three lines give the extra scatter for the 3-, 5- and 7-point stencils on the
same quantized data. Does a wider stencil help? What does that tell you about which error source you
are looking at? *(two or three sentences)*

*Your answer:*

---
## 6 · ⭐ YOUR TURN — the alignment helper

One more function, and it is the last code you write today. In the assignment the kinetic energy
comes from a derivative and is *short*, while the potential energy comes straight from position and
is *full length*. `trim_edges(arr, n)` drops `n` points from each end of an array, which is what
puts PE and $t$ onto the same time axis as KE.

One line — but mind the `n = 0` case. `arr[0:-0]` is not `arr`; it is empty. The last test below is
there to catch exactly that.

In [ ]:
def trim_edges(arr, n):
    '''Drop n points from each end of arr.'''
    # ⭐ YOUR CODE HERE -- one line. Delete the raise below.
    raise NotImplementedError('drop n points from each end')


# n = 0 must give the array back untouched -- arr[0:-0] is empty, not arr
assert len(trim_edges(t, 0)) == len(t), 'trim_edges(arr, 0) must return arr unchanged'

# and it must line up with central_diff for every stencil
y = f(t)
for s in (3, 5, 7):
    k  = (s - 1)//2
    d  = central_diff(y, h, s)
    tc = trim_edges(t, k)
    assert len(d) == len(tc), f'stencil {s}: {len(d)} vs {len(tc)}'
    print(f'stencil {s}: derivative and trimmed time axis both length {len(d)}  OK')

---
## 7 · Your plan for the assignment

`central_diff` is written and tested, so the core numerics are done.
You will apply them on the assignment.

Open Assignment 1 on D2L and skim it — sections 1 to 4 are the code.

**✍ Q7.1** What else do you have to write? List the functions you still need. Beside each, say how
you will know it works, *before* you build the next thing on top of it. A short phrase is enough.
The first row is filled in as an example — it is what you did in Section 1.

| What I still have to write | How I will know it works |
|---|---|
| `central_diff` — *already done* | *right output length; exact on a quadratic* |
| | |
| | |
| | |

**✍ Q7.2** What is your plan for AI? LLMs are permitted on this assignment. That makes *how* you
use them a decision worth making now, before you start.

Split the assignment into three piles, and put at least one thing in each. One line per cell.

| | Which part of the assignment | Role-pair | How I check the result |
|---|---|---|---|
| **I write it myself** | | — | |
| **I work on it with the LLM** | | | |
| **The LLM drafts it, I verify** | | | |

Two reminders from Unit 0 as you fill that in:

* With a deliverable in front of you, the **default** is human-manager / AI-worker. That is the
  productivity end of the axis, not the learning end. If that is what you want for a given piece,
  choose it on purpose. Coach/player, tutor/student and author/critic put the work back on you.
* The third column is the one people skip. "I read it and it looked right" is not a check. You have
  spent today building actual checks — a known answer, an array length, an expected order — and the
  same kinds of check work on code you did not write.

Section 6 of the assignment asks you to log what you *actually* did with an LLM. When you get
there, compare it against what you have just written.

**✍ Q7.3** You will plot $E(t)$ and it will not be flat. Name the three things that could be responsible, and
say which one you expect to survive to the very end. *(three names and a sentence)*

*Your answer:*

### Before you hand in

Check that all of these ran without error:

- [ ] `make_grid` passes its five tests in Section 0
- [ ] `central_diff` passes the length test and the quadratic test in Section 1
- [ ] `max_error` reproduces the number Section 2 printed
- [ ] the error ratios in Section 3 come out at about 4
- [ ] `central_diff` works for stencils 3, 5 and 7, and passes the constant test
- [ ] a higher-order stencil does **not** fix quantization
- [ ] `trim_edges` lines up with `central_diff` for every stencil

**Save this notebook and hand it in.** `central_diff` and `trim_edges` go straight into
Assignment 1 — they are done.